# LegalMitra: Llama-3.1-8B-Instruct 4-bit QLoRA Fine-Tuning
### Native HuggingFace PEFT + BitsAndBytes Pipeline (Python 3.13 Ready)

This notebook fine-tunes Meta-Llama-3.1-8B-Instruct on the LegalMitra Indian Legal IRAC instruction dataset using native HuggingFace `transformers`, `peft`, and `bitsandbytes`.

## 1. Install Standard Dependencies

In [ ]:
!pip install -q -U transformers accelerate bitsandbytes peft datasets
import torch
print('PyTorch Version:', torch.__version__)
print('CUDA Available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('Device:', torch.cuda.get_device_name(0))


## 2. Load Base Model in 4-bit NF4 Quantization

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import prepare_model_for_kbit_training, LoraConfig, get_peft_model

model_id = 'unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit'

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16,
)

tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'right'

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map='auto'
)

model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
    lora_dropout=0.05,
    bias='none',
    task_type='CAUSAL_LM'
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
print('Base Llama-3.1-8B model and LoRA adapter initialized successfully!')


## 3. Load & Format Dataset

In [ ]:
import os
import glob
from datasets import load_dataset

# Dynamically locate the dataset in /kaggle/input/
train_candidates = glob.glob('/kaggle/input/**/legal_instructions_train.jsonl', recursive=True)
val_candidates = glob.glob('/kaggle/input/**/legal_instructions_val.jsonl', recursive=True)

if not train_candidates:
    raise FileNotFoundError('Could not find legal_instructions_train.jsonl in /kaggle/input/')

train_file = train_candidates[0]
val_file = val_candidates[0] if val_candidates else None
print(f'Using train file: {train_file}')
print(f'Using val file: {val_file}')

data_files = {'train': train_file}
if val_file:
    data_files['validation'] = val_file

dataset = load_dataset('json', data_files=data_files)
print(dataset)

# Format as Llama-3.1 chat prompt
def format_llama_chat(example):
    instruction = example.get('instruction', '')
    context = example.get('context', '')
    response = example.get('response', '')
    
    user_content = f'{instruction}\n\nLegal Evidence and Precedents:\n{context}' if context else instruction
    
    text = (
        '<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\n'
        'You are LegalMitra AI, an expert Indian legal assistant. You answer queries using IRAC methodology grounded strictly in the provided statutory provisions and court precedents.<|eot_id|>'
        f'<|start_header_id|>user<|end_header_id|>\n\n{user_content}<|eot_id|>'
        f'<|start_header_id|>assistant<|end_header_id|>\n\n{response}<|eot_id|>'
    )
    return {'text': text}

dataset = dataset.map(format_llama_chat)
print('Sample prompt:')
print(dataset['train'][0]['text'][:600] + '...')


## 4. Tokenize Dataset

In [ ]:
max_seq_length = 2048

def tokenize_function(examples):
    result = tokenizer(
        examples['text'],
        truncation=True,
        max_length=max_seq_length,
        padding=False
    )
    result['labels'] = result['input_ids'].copy()
    return result

tokenized_dataset = dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=dataset['train'].column_names
)
print('Tokenization complete!')


## 5. Configure Trainer and Train QLoRA Adapter

In [ ]:
from transformers import TrainingArguments, Trainer, DataCollatorForSeq2Seq

training_args = TrainingArguments(
    output_dir='./legalmitra_lora_output',
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    warmup_steps=10,
    max_steps=250,
    learning_rate=2e-4,
    fp16=True,
    logging_steps=10,
    optim='paged_adamw_8bit',
    weight_decay=0.01,
    lr_scheduler_type='linear',
    seed=42,
    save_strategy='steps',
    save_steps=100,
    save_total_limit=2,
    report_to='none'
)

trainer = Trainer(
    model=model,
    train_dataset=tokenized_dataset['train'],
    eval_dataset=tokenized_dataset.get('validation'),
    args=training_args,
    data_collator=DataCollatorForSeq2Seq(tokenizer, pad_to_multiple_of=8, return_tensors='pt', padding=True),
)

print('Starting fine-tuning...')
trainer_stats = trainer.train()
print('Training complete!')


## 6. Save LoRA Adapter and Package for LegalMitra

In [ ]:
import shutil
import zipfile

final_save_dir = './legalmitra_llama3_lora_final'
model.save_pretrained(final_save_dir)
tokenizer.save_pretrained(final_save_dir)
print(f'Saved LoRA adapter weights and tokenizer to {final_save_dir}')

# Package into downloadable zip
zip_filename = '/kaggle/working/legalmitra_llama3_lora.zip'
with zipfile.ZipFile(zip_filename, 'w', zipfile.ZIP_DEFLATED) as zipf:
    for root, dirs, files in os.walk(final_save_dir):
        for file in files:
            full_path = os.path.join(root, file)
            rel_path = os.path.relpath(full_path, final_save_dir)
            zipf.write(full_path, rel_path)

zip_size_mb = os.path.getsize(zip_filename) / (1024 * 1024)
print(f'Successfully created: {zip_filename} ({zip_size_mb:.2f} MB)')
print('You can now download this zip file from the Kaggle Output panel!')
